# 44. SQLAlchemy ORM — `sqlalchemy`
**Python Libraries Lab · Phase 3 · Databases**

_Work with databases using Python classes instead of raw SQL._

✅ **Runs in Colab** as-is.

SQLAlchemy is the most widely used database toolkit in Python. Its ORM, object relational mapper, lets you describe tables as Python classes and rows as objects. You add a customer object, commit, and SQLAlchemy writes the SQL. The same code works with SQLite, PostgreSQL, MySQL and others, you just change the connection string.

**In this notebook:** You will model ShopEasy customers and orders with a relationship between them, create the tables, add data, query with filters, joins and group by, update and delete safely, look at the SQL that SQLAlchemy writes, and load a query straight into pandas.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q sqlalchemy
import importlib.metadata as md
for pkg in ['sqlalchemy']:
    print(pkg, md.version(pkg))

## Step 1 · Engine and models
Two tables, linked: a customer has many orders. `echo=False` hides the SQL (we will look at it later).

In [ ]:
from sqlalchemy import create_engine, ForeignKey, String, select, func
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

engine = create_engine("sqlite:///shop_orm.db", echo=False)

class Base(DeclarativeBase):
    pass

class Customer(Base):
    __tablename__ = "customers"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    city: Mapped[str] = mapped_column(String(30))
    orders: Mapped[list["Order"]] = relationship(back_populates="customer", cascade="all, delete-orphan")
    def __repr__(self):
        return f"Customer({self.name!r}, {self.city!r})"

class Order(Base):
    __tablename__ = "orders"
    id: Mapped[int] = mapped_column(primary_key=True)
    amount: Mapped[float]
    customer_id: Mapped[int] = mapped_column(ForeignKey("customers.id"))
    customer: Mapped[Customer] = relationship(back_populates="orders")

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
print("Tables:", list(Base.metadata.tables))

## Step 2 · Add data in a session
Create objects, attach orders to customers, then commit once.

In [ ]:
with Session(engine) as s:
    asha = Customer(name="Asha", city="Pune", orders=[Order(amount=499), Order(amount=2999)])
    ravi = Customer(name="Ravi", city="Delhi", orders=[Order(amount=1299)])
    meera = Customer(name="Meera", city="Pune", orders=[Order(amount=899), Order(amount=149), Order(amount=650)])
    john = Customer(name="John", city="Mumbai")
    s.add_all([asha, ravi, meera, john])
    s.commit()
    print("Saved. Asha has", len(asha.orders), "orders")

## Step 3 · Query with select()
Filters, sorting, and following relationships.

In [ ]:
with Session(engine) as s:
    pune = s.scalars(select(Customer).where(Customer.city == "Pune").order_by(Customer.name)).all()
    print("Pune customers:", pune)
    big = s.scalars(select(Order).where(Order.amount > 800)).all()
    print("Big orders:", [(o.customer.name, o.amount) for o in big])

## Step 4 · Join and group by
Revenue per customer in one query. `outerjoin` keeps John, who has no orders.

In [ ]:
stmt = (select(Customer.name, func.count(Order.id).label("orders"), func.coalesce(func.sum(Order.amount), 0).label("revenue"))
        .outerjoin(Customer.orders).group_by(Customer.id).order_by(func.sum(Order.amount).desc()))
with Session(engine) as s:
    for name, n, revenue in s.execute(stmt):
        print(f"{name:<6} {n} orders  Rs.{revenue:,.0f}")
print("\nThe SQL SQLAlchemy wrote:\n", stmt)

## Step 5 · Update and delete
Change objects and commit. Deleting a customer also deletes their orders (the `cascade` rule).

In [ ]:
with Session(engine) as s:
    ravi = s.scalars(select(Customer).where(Customer.name == "Ravi")).one()
    ravi.city = "Gurugram"
    meera = s.scalars(select(Customer).where(Customer.name == "Meera")).one()
    s.delete(meera)
    s.commit()
    print("Customers:", s.scalars(select(Customer)).all())
    print("Orders left:", s.scalar(select(func.count(Order.id))))

## Step 6 · Straight into pandas
`read_sql` runs a query and returns a DataFrame.

In [ ]:
import pandas as pd
pd.read_sql(select(Customer.name, Customer.city, Order.amount).join(Customer.orders), engine)

## ✍️ Your turn
Store the **total amount of all orders** still in the database (a number) in `answer`, using `select(func.sum(Order.amount))`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 4797, f"Expected 4797, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
with Session(engine) as s:
    answer = s.scalar(select(func.sum(Order.amount)))

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Engine**
- **Session**
- **ORM model**
- **Relationship**
- **select()**
- **Migration**

**Next:** 45. PostgreSQL & MySQL (`psycopg2 · pymysql`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/45_psycopg2_pymysql.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install sqlalchemy
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.